In [4]:
import sys
print(sys.executable)

c:\Users\Kevin\miniforge3\envs\aikb\python.exe


In [5]:
%pip install sentence-transformers pandas scikit-learn matplotlib plotly jieba

  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached plotly-6.8.0-py3-none-any.whl.metadata (9.0 kB)
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ----------------------- --------------- 11.5/19.2 MB 60.0 MB/s eta 0:00:01
     ---------------------------------------- 19.2/19.2 MB 55.1 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.26.8-py3-none-any.whl.metadata (15 kB)
  Using cached safeten

In [ ]:
import sentence_transformers, pandas, sklearn
print("environment ready")

c:\Users\Kevin\miniforge3\envs\aikb\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


环境就绪


# 阶段 1：AI 使用画像（时间序列 EDA）

先用合成数据练手，等真实导出准备好了随时可以换成 `src/parsing/chatgpt_parser.py` 的输出。

目标表结构和解析器产出的一致：`conversation_id, turn, role, timestamp, text`。

第一步：造一份"看起来像真实使用习惯"的假数据——故意加入夜间高峰、全年增长趋势这些模式，这样后面统计出来的图才有东西可看，不是纯噪声。

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

N_CONVERSATIONS = 150
DATE_RANGE = pd.date_range("2025-01-01", "2025-12-31", freq="D")

# 模拟"夜猫子型"用户：晚上9点到凌晨1点是高峰，白天工作时段较低
HOUR_WEIGHTS = np.array([
    0.5, 0.3, 0.2, 0.1, 0.1, 0.1, 0.2, 0.3,
    0.8, 1.0, 1.0, 0.9, 0.8, 0.9, 1.0, 1.0,
    1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 2.5, 1.5,
])
HOUR_WEIGHTS = HOUR_WEIGHTS / HOUR_WEIGHTS.sum()

# 全年使用量缓慢增长的趋势
DAY_WEIGHTS = np.linspace(0.5, 1.5, len(DATE_RANGE))
DAY_WEIGHTS = DAY_WEIGHTS / DAY_WEIGHTS.sum()

TOPIC_PHRASES = {
    "代码调试": ["这段代码报错了", "为什么会抛异常", "帮我看看这个bug", "怎么优化这个函数"],
    "写作润色": ["帮我润色这段话", "这句话怎么表达更自然", "帮我写一段开头"],
    "求职规划": ["这份简历怎么改", "面试官可能会问什么", "要不要跳槽"],
    "概念学习": ["能不能讲讲这个原理", "这个公式怎么推导", "这两个概念有什么区别"],
}
topics = list(TOPIC_PHRASES.keys())

rows = []
for conv_id in range(1, N_CONVERSATIONS + 1):
    day = rng.choice(DATE_RANGE, p=DAY_WEIGHTS)
    hour = rng.choice(24, p=HOUR_WEIGHTS)
    minute = rng.integers(0, 60)
    t = pd.Timestamp(day) + pd.Timedelta(hours=int(hour), minutes=int(minute))

    topic = rng.choice(topics)
    n_turns = rng.integers(2, 13)  # 2~12条消息，user/assistant交替

    for turn in range(n_turns):
        role = "user" if turn % 2 == 0 else "assistant"
        phrase = rng.choice(TOPIC_PHRASES[topic])
        rows.append({
            "conversation_id": f"conv_{conv_id:04d}",
            "turn": turn,
            "role": role,
            "timestamp": t,
            "text": f"{phrase}（{topic}，第{turn + 1}轮）",
        })
        t += pd.Timedelta(seconds=int(rng.integers(10, 300)))

df = pd.DataFrame(rows)
df.shape

(1050, 5)